# Lab 01 — Python Refresher for Data People
**Junior Analyst Track** · Beginner · ~45 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Map Excel columns/rows/sheets to Python lists, dicts, and lists-of-dicts
2. Clean messy text and numbers with strip/title and None handling
3. Build reusable cleaning functions and comprehensions
4. Answer revenue-by-region from a 12-row messy sales sheet

## Datasets (this folder)
- `sales_spreadsheet.csv` — **upload** in Colab (or keep next to the notebook locally)

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-01-python-refresher-for-data-people/lab-01-python-refresher-for-data-people.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`sales_spreadsheet.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-01-python-refresher-for-data-people"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/labs/lab-01-python-refresher-for-data-people/bundle/dataset.zip"
NEED = ["sales_spreadsheet.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Junior Analyst Track: Cleaning a Sales Spreadsheet

> **Scenario:** You're a junior analyst. Someone sends you `sales_spreadsheet.csv` — 12 rows, messy product names, missing quantities, inconsistent regions. Your job: clean it with pure Python and answer: *what's our total revenue by region?*
>
> **You will learn:** lists, dicts, loops, functions, comprehensions — all with sales-data examples.
> **Time:** ~45 minutes. **Level:** Beginner. **Needs:** Python 3.8+ only (no pandas needed).

### Excel → Python mental map

Why it matters: almost every cleanup task you meet as an analyst already has an Excel shape. Translating that shape into list/dict/loop means you can reuse instincts from SUMIF, VLOOKUP, and blank-cell formulas instead of learning an entirely new mental model.

| Excel idea | Python idea | Example |
|---|---|---|
| A column (e.g. all Prices) | a `list` | `[899.99, 25.5, 75.0]` |
| One row (e.g. Order 101) | a `dict` | `{"OrderID": 101, "Product": "Laptop"}` |
| Whole sheet | a `list of dicts` | `[{"OrderID":101, ...}, {"OrderID":102, ...}]` |
| `=IF(ISBLANK(A2),0,A2)` | conditional expression with `None` check | `qty if qty is not None else 0` |
| `=SUMIF(Region,"West",Total)` | loop + dict `.get` accumulator | `totals[region] = totals.get(region, 0) + total` |

Keep that map in mind — every numbered section below is one Excel habit restated in Python, and the exercises ask you to combine them on the real 12-row file.

### Setup: load the real dataset

Why it matters for analysts: the load step is where dirty data either becomes safe types or silently poisons every later calculation. Cell 0 already downloaded + unzipped `sales_spreadsheet.csv`. Load it below with the `csv` module — blanks become `None`, numbers become `int`/`float`. From here on, `sales` **is** the dataset file, and every exercise runs against it, so any number you report should be reproducible from these 12 rows.

In [ ]:
# Load the real dataset file (Cell 0 fetched it via wget + unzip)
import csv

def _int_or_none(v):
    v = v.strip()
    return int(v) if v else None

def _float_or_none(v):
    v = v.strip()
    return float(v) if v else None

with open("sales_spreadsheet.csv", newline="", encoding="utf-8") as f:
    sales = [
        {
            "OrderID": int(r["OrderID"]),
            "Product": r["Product"],
            "Quantity": _int_or_none(r["Quantity"]),
            "Price": _float_or_none(r["Price"]),
            "Region": r["Region"],
        }
        for r in csv.DictReader(f)
    ]

print(len(sales), "rows from sales_spreadsheet.csv")  # 12
print(sales[0])
# {'OrderID': 101, 'Product': '  Laptop ', 'Quantity': 2, 'Price': 899.99, 'Region': 'West'}


Messy on purpose. Notice: extra spaces, ALL CAPS, empty string `""`, `None` for missing, `west` vs `West` vs `WEST`.

**What to notice:**
- The loader prints `12 rows from sales_spreadsheet.csv` — if you ever see a different count, the file path or download step is wrong before any cleaning logic runs.
- `sales[0]` shows `Product: '  Laptop '` with leading/trailing spaces intact, so raw text is never "already clean" even when it looks readable.
- `Quantity` and `Price` use the `_int_or_none` / `_float_or_none` helpers: blank cells become `None`, not `0` or `""` — that distinction drives every later filter.
- Regions arrive as raw variants (`west` / `West` / `WEST`); normalisation is a deliberate later step, not something `csv.DictReader` does for you.

> **Pitfall:** Never do arithmetic directly on a raw cell that might be missing. `None * 2` raises `TypeError`, while `0 * 2` silently returns `0` and pretends the row was real data. Always branch on `is None` (or coerce with an explicit default) before multiplying quantity by price.

---

## 1. Lists — your columns

Why it matters for analysts: a list is an ordered collection of values — think of one Excel column such as all Prices or all Product names. Row counts (`len`), column totals (`sum`), and min/max checks are the first sanity pass you run on any export, and in Python they map one-to-one onto list built-ins.

In [ ]:
prices = [899.99, 25.50, 75.00, 12.99]
products = ["Laptop", "Mouse", "Keyboard"]

# Indexing (0-based!) and slicing
print(prices[0])      # 899.99 — first item
print(prices[-1])     # 12.99 — last item
print(products[0:2])  # ['Laptop', 'Mouse'] — slice, end is exclusive

# Useful list ops for analysts
print(len(prices))    # 4 — row count
print(sum(prices))    # 1013.48 — like =SUM()
print(min(prices), max(prices))  # 12.99 899.99 — like =MIN(), =MAX()
print(sorted(products))  # alphabetical copy


**What to notice:**
- `prices[0]` is `899.99` and `prices[-1]` is `12.99` — Python counts from the left starting at 0, but `-1` is a reliable "last cell" shortcut (no need for `len(prices)-1` every time).
- `products[0:2]` returns only `['Laptop', 'Mouse']`: the start index is included, the end index is excluded, so a slice of length 2 uses `0:2`, not `0:1`.
- `len(prices)` is `4` and `sum(prices)` is `1013.48` — the same mental operations as counting rows and `=SUM()` over a column, with no header row to skip.
- `min`/`max` print `12.99` and `899.99` in one call, and `sorted(products)` returns a new alphabetical list without mutating `products`.

> **Pitfall:** Slice ends are exclusive, so `products[0:2]` gives two items, not three. Getting this wrong is the classic off-by-one: to grab Excel rows 2–4 (three rows) in a zero-based list you need indices `1:4`, not `1:3`.

Mutating a list (cleaning in place): cleaning often means walking a messy column once, trimming each value with `.strip()`, and rebuilding a tidy list. The loop below is the explicit version; Section 5 shows the same idea as a one-line comprehension.

In [ ]:
raw_products = ["  Laptop ", "KEYBOARD", "mouse", ""]
cleaned = []
for p in raw_products:
    cleaned.append(p.strip())  # strip() removes spaces
print(cleaned)  # ['Laptop', 'KEYBOARD', 'mouse', '']

# Remove empties, normalise case
cleaned = [p.strip().title() for p in cleaned if p.strip() != ""]
print(cleaned)  # ['Laptop', 'Keyboard', 'Mouse']
# .title() -> 'KEYBOARD' becomes 'Keyboard'. Perfect for product names.


**What to notice:**
- After `.strip()` alone, `cleaned` is `['Laptop', 'KEYBOARD', 'mouse', '']` — spaces are gone, but case is still inconsistent and the blank cell remains as `''`.
- The comprehension rebuilds the list as `['Laptop', 'Keyboard', 'Mouse']`: the `if p.strip() != ""` clause drops empty cells *before* `.title()` runs on survivors, so blanks never become a phantom `""` entry in your category counts.
- Original messy values are not repaired inside `raw_products`; cleaning builds a new list, which is safer when you still need to audit the raw file.

> **Pitfall:** Filter on the *stripped* value, not the raw one. A cell like `"   "` looks non-empty in Python (`"   " != ""` is `True`) but is blank in Excel after trim — `if p.strip() != ""` handles that; `if p != ""` does not.

**Key methods to memorise:** `.append(x)`, `.remove(x)`, `len()`, `sum()`, `sorted()`, `list.count(x)`.

> Try it: what does `prices + [49.99]` do? (Answer: returns a new longer list, original unchanged — concatenation never edits in place, unlike `.append`.)

---

## 2. Dicts — your rows

Why it matters for analysts: a dict maps labelled fields to values — exactly like one spreadsheet row where the headers become keys. Grouping, VLOOKUP-style lookups, and "add a calculated column" are all dict operations, so if you can think in rows-with-header-names, you already think in dicts.

In [ ]:
order = {"OrderID": 101, "Product": "  Laptop ", "Quantity": 2, "Price": 899.99, "Region": "West"}

print(order["Product"])          # '  Laptop ' — direct access (errors if key missing)
print(order.get("Discount", 0))  # 0 — safe access with default, never crashes

order["Product"] = order["Product"].strip().title()  # clean that cell
order["Total"] = order["Quantity"] * order["Price"]  # add a calculated column
print(order)
# {'OrderID': 101, 'Product': 'Laptop', 'Quantity': 2, 'Price': 899.99, 'Region': 'West', 'Total': 1799.98}

print(list(order.keys()))    # headers
print(list(order.values()))  # row values
print(list(order.items()))   # pairs — you'll loop over these a lot


**What to notice:**
- `order["Product"]` initially prints `'  Laptop '` — dict access does **not** clean anything; the spaces you loaded from the CSV are still there until you assign a cleaned value back.
- `order.get("Discount", 0)` returns the default `0` because no such header exists in this row; the same lookup with `order["Discount"]` would raise `KeyError` and kill the notebook cell.
- After cleaning, `Total` is `1799.98` (`2 * 899.99`) — a calculated column is just a new key, the same way you'd fill a helper column beside the table in Excel.
- `list(order.keys())` gives you the header list; `list(order.items())` gives `(key, value)` pairs, which is the shape you want when looping rows in Section 3.

> **Pitfall:** Use `order["Key"]` only when the key must exist; use `order.get("Key", default)` for optional columns (Discount, Notes) that are blank or absent on some rows. Mixing them up is the dict equivalent of a broken VLOOKUP: one missing header stops the whole run.

A whole worksheet is just a list of those row dicts — and you already loaded one as `sales`. Indexing picks a row; a tiny loop stands in for `=VLOOKUP(OrderID, …)`:

In [ ]:
print(sales[0])              # first row (Order 101)
print(sales[0]["Region"])    # 'West'
print(len(sales))            # 12 rows

# Like =VLOOKUP: find Order 108
order_108 = None
for row in sales:
    if row["OrderID"] == 108:
        order_108 = row
        break
print(order_108)


**What to notice:**
- `len(sales)` is `12` — the same row count printed at load time, confirming you're still on the full file, not a filtered slice.
- `sales[0]["Region"]` is `'West'`: list index first (which row), dict key second (which column), matching Excel's "row number then header" habit in reverse order of cell references.
- The VLOOKUP-style loop stops with `break` once `OrderID == 108` is found; if no match existed, `order_108` would stay `None`, which is why the variable is initialised before the loop instead of being created only on a hit.

> **Pitfall:** Don't search a list of dicts by writing `sales[108]` — that is "the 108th row," not "the row whose OrderID is 108." Order IDs are values inside each dict; positional indexing only works when you know the physical row number.

---

## 3. Loops — your cleaning engine

Why it matters for analysts: almost every spreadsheet task you'd build with a filled-down formula or a PivotTable is a loop over rows in Python — read the row, decide, accumulate. The three patterns below (sum with skips, enumerate with row numbers, count-by-group) cover the majority of day-one cleanup jobs on `sales_spreadsheet.csv`.

### 3a. `for` loop (you'll use this 95% of the time)

In [ ]:
# Pattern 1: total revenue, skipping bad rows
total = 0
for row in sales:
    qty = row["Quantity"]
    price = row["Price"]
    if qty is None or price is None:
        continue  # skip incomplete rows, like filtering blanks in Excel
    total += qty * price
print(f"Total (skip missing): ${total:.2f}")

# Pattern 2: enumerate when you need row numbers
for i, row in enumerate(sales, start=2):  # start=2 mimics Excel (header is row 1)
    if not str(row["Product"]).strip():
        print(f"Row {i}: missing product (Order {row['OrderID']})")

# Pattern 3: counting by group (like a PivotTable COUNTIF)
region_counts = {}
for row in sales:
    region = str(row["Region"]).strip().title()  # 'west' -> 'West', ' WEST' -> 'West'
    region_counts[region] = region_counts.get(region, 0) + 1
print(region_counts)  # {'West': 4, 'East': 4, 'North': 2, 'South': 1}


**What to notice:**
- Pattern 1 only adds to `total` when **both** `qty` and `price` are not `None`, so missing cells are excluded the same way filtering blanks out of a SUM range excludes them — the printed total never includes a phantom `0 * price` row.
- Pattern 2 starts `enumerate` at `2` so printed row numbers line up with Excel (header is spreadsheet row 1); blank product cells surface as `Row N: missing product (Order …)` messages you can hand back to whoever owns the file.
- Pattern 3 normalises with `.strip().title()` **before** using the region as a key, which is why the counter shows tidy buckets `{'West': 4, 'East': 4, 'North': 2, 'South': 1}` instead of separate entries for `'west'`, `'West'`, and `' WEST'`.
- The `.get(region, 0) + 1` idiom is the PivotTable COUNTIF pattern: first time a region appears the default `0` is used, every later occurrence adds one.

> **Pitfall:** `None` and `0` are not interchangeable inside a loop. Skipping with `continue` when a value is `None` (Pattern 1) keeps bad rows out of the sum; accidentally substituting `0` would still "work" but would let a row with a real price and missing quantity contribute `0` and quietly change your valid-row counts. Decide deliberately: skip, flag, or coerce — never mix the three across the same report.

### 3b. `if / elif / else` inside loops

In [ ]:
for row in sales:
    raw = str(row["Product"]).strip()
    if raw == "":
        status = "NEEDS PRODUCT"
    elif row["Quantity"] is None or row["Price"] is None:
        status = "NEEDS NUMBERS"
    else:
        status = "OK"
    print(row["OrderID"], status)


**What to notice:**
- Each row prints exactly one label because `if / elif / else` checks in order: a blank product is reported as `NEEDS PRODUCT` even if its numbers are also missing — you never see two statuses for one OrderID.
- The empty-product test runs on `str(row["Product"]).strip()`, so both a truly empty cell and a spaces-only cell land in `NEEDS PRODUCT` rather than slipping through as a real product name.
- `NEEDS NUMBERS` fires only when the product text survived but `Quantity is None` or `Price is None` — matching the loader's decision to keep missing numerics as `None` instead of `0`.

> **Pitfall:** `strip()` text first, *then* compare — `" west " == "west"` is `False`, and so is `"WEST" == "West"`. A classic junior bug is flagging rows as dirty (or clean) based on untrimmed comparisons; normalise case and spaces before any `==` test on free-text columns.

---

## 4. Functions — reusable cleaning steps

Why it matters for analysts: if you copy-paste cleaning logic twice, make it a function. A named `clean_text` / `clean_row` / `revenue_by_region` pipeline is how you turn a one-off notebook fix into something a teammate can rerun next month on a new export without re-reading every loop.

In [ ]:
def clean_text(value):
    """Strip spaces and title-case. Empty/None -> ''."""
    if value is None:
        return ""
    return str(value).strip().title()


def clean_row(row):
    """Return a cleaned copy of a sales row + computed Total. Never mutates input."""
    cleaned = {
        "OrderID": row["OrderID"],
        "Product": clean_text(row["Product"]),
        "Quantity": row["Quantity"] if row["Quantity"] is not None else 0,
        "Price": row["Price"] if row["Price"] is not None else 0.0,
        "Region": clean_text(row["Region"]),
    }
    cleaned["Total"] = round(cleaned["Quantity"] * cleaned["Price"], 2)
    return cleaned


def revenue_by_region(rows):
    """Sum Total per Region. Expects already-cleaned rows."""
    totals = {}
    for r in rows:
        totals[r["Region"]] = totals.get(r["Region"], 0) + r["Total"]
    return totals


# Use them:
clean_sales = [clean_row(r) for r in sales]  # preview of next section!
print(clean_sales[0])
# {'OrderID': 101, 'Product': 'Laptop', 'Quantity': 2, 'Price': 899.99, 'Region': 'West', 'Total': 1799.98}

print(revenue_by_region(clean_sales))


**What to notice:**
- `clean_sales[0]` mirrors the raw first row but with `'  Laptop '` rewritten to `'Laptop'` and a computed `Total` of `1799.98` — same OrderID 101, now safe to aggregate.
- `clean_text(None)` returns `''` rather than crashing on `.strip()`, which is why blank products become a consistent empty string you can filter with `!= ""` later.
- Inside `clean_row`, missing `Quantity`/`Price` become `0`/`0.0`, so `Total` for incomplete rows is `0.0` instead of raising `TypeError` — and `revenue_by_region` sums those zeros harmlessly while Section 5's `Total > 0` filter can still drop them from "valid order" counts.
- `clean_row` builds and returns a **new** dict (`cleaned = {...}`); the original row in `sales` keeps its messy values, so you can always compare raw vs cleaned during QA.

Why functions help analysts:
- `clean_text` fixes *every* text column the same way — Product, Region, and any future free-text header get identical strip/title rules instead of ad-hoc edits per cell.
- `clean_row` is testable: feed one row, check output; regression-test it when the export format shifts.
- Default pattern `x if x is not None else 0` safely fills blanks (like Excel `=IF(ISBLANK(A2),0,A2)`) **after** you've decided that blank means zero for this metric.

> **Pitfall:** Coercing `None → 0` in `clean_row` is right for revenue (a missing price contributes nothing) but wrong if you later ask "how many rows had missing price?" — those rows now look like real zeros. Keep a separate validity check (`row["Price"] is not None` on the raw data, or `Total > 0` / a dedicated flag on cleaned data) whenever the question is about *data quality* rather than *totals*.

---

## 5. Comprehensions — one-line transforms

Why it matters for analysts: comprehensions are compact loops for building lists and dicts. After you've written the longhand version in Sections 1 and 3, the same filters and group-bys collapse into one readable line — which is how most analysts do column transforms once the logic is already proven.

In [ ]:
# List comprehension: [expression for item in iterable if condition]

# All totals in one line
totals = [r["Quantity"] * r["Price"] for r in sales
          if r["Quantity"] is not None and r["Price"] is not None]
print(totals)

# Clean product list, dropping blanks — compare to the loop in Section 1
products_clean = [r["Product"].strip().title() for r in sales if str(r["Product"]).strip()]
print(products_clean)

# Dict comprehension: {key_expr: value_expr for item in iterable}
# Revenue per order id
revenue_per_order = {r["OrderID"]: (r["Quantity"] or 0) * (r["Price"] or 0) for r in sales}
print(revenue_per_order[101])  # 1799.98
# (r["Quantity"] or 0) trick: None, 0, '' all become 0. Handy for messy sheets.

# Full clean pipeline in 2 lines (uses functions from Section 4)
clean_sales = [clean_row(r) for r in sales]
valid_sales = [r for r in clean_sales if r["Product"] != "" and r["Total"] > 0]
print(f"{len(valid_sales)} valid orders out of {len(sales)}")
print(revenue_by_region(valid_sales))


**What to notice:**
- The first comprehension only emits a total when `Quantity is not None and Price is not None`, so `totals` has one entry per *complete* row — the same guard Pattern 1 used with `continue`, just expressed as an `if` clause.
- `products_clean` uses `if str(r["Product"]).strip()` as its filter: blank and whitespace-only names never make the list, and every survivor is `.strip().title()`-ed in the same expression (transform + filter in one line).
- `revenue_per_order[101]` prints `1799.98`, matching `clean_row`'s Total for Order 101 — two different styles (dict-comprehension with `or 0` vs function pipeline) agreeing on the same cell of output.
- The final two prints count valid orders and group revenue by region after `clean_row`; rows with an empty product or `Total <= 0` are excluded before aggregation, so the region dict only reflects sellable lines.

When *not* to use a comprehension: if the logic needs 3+ `if`s, side effects (printing, appending elsewhere), or you'd need a comment to explain the condition — use a plain loop for readability.

> **Pitfall:** The `(r["Quantity"] or 0)` trick treats **any** falsy value as missing: `None`, `0`, `''`, and even `0.0` all become `0`. That's fine for revenue on this messy sheet, but it also means a legitimate zero quantity is indistinguishable from a blank. When the analysis cares about "was this cell missing?", go back to explicit `is None` checks as in Section 3.

---

## Putting it together

Why it matters for analysts: real requests arrive as "clean this file and tell me revenue by region," not as isolated syntax drills. The `analyse` function chains everything you've built — clean, filter, aggregate, report — into one call that takes raw rows and returns a JSON-ready summary a manager can read.

In [ ]:
def analyse(sales_rows):
    cleaned = [clean_row(r) for r in sales_rows]
    valid = [r for r in cleaned if r["Product"] and r["Total"] > 0]
    return {
        "rows_in": len(sales_rows),
        "rows_valid": len(valid),
        "rows_dropped": len(sales_rows) - len(valid),
        "total_revenue": round(sum(r["Total"] for r in valid), 2),
        "by_region": revenue_by_region(valid),
    }

import json
print(json.dumps(analyse(sales), indent=2))
# Expect ~ total 4451.83, West highest. Run it!


**What to notice:**
- `rows_in` is always `12` for the unfiltered file; `rows_valid` + `rows_dropped` must add back to `12`, which is the first arithmetic check to run on any pipeline that drops rows.
- `total_revenue` should come out near `4451.83` (as the comment notes) — if your number differs, a clean/filter step is over- or under-including rows rather than a summation bug.
- `by_region` is keyed by cleaned region names, and the expected shape has West highest; compare that ranking against a quick manual SUMIF in the spreadsheet when validating.
- The result is plain dicts/lists/floats, so `json.dumps(..., indent=2)` works without any special DataFrame-to-JSON conversion.

---

## Exercises (do these!)

### Exercise 1 — High-value orders (lists + loops)
Using `sales` (raw), print `OrderID` and `Total` for every order where `Quantity` and `Price` exist and `Total > 500`. Also print how many such orders exist.
*Expected: 2 orders (101 and 106). Hint: `if qty is not None and price is not None:` then compute.*

**Follow-up:** What share of valid revenue comes from those high-value orders? Check: about 0.7184 (two orders drive ~72%).

<details>
<summary>Hint</summary>

Loop, compute `qty * price`, collect matches in a list, then print.
</details>

### Exercise 2 — Category counter (dicts + functions)
Write a function `category(product)` that returns `"Computer"` for Laptop/Monitor/Keyboard/Mouse (any case/spaces), `"Audio"` for Headset, else `"Other"` (and `"Unknown"` for blank). Then loop over `sales`, clean each product with `clean_text`, categorise it, and build a dict `{category: count}`.
*Expected counts approx: Computer 10, Audio 1, Unknown 1. Hint: reuse `clean_text` from Section 4.*

**Follow-up:** What share of the 12 orders are Computers? Check: 10/12, about 0.8333.

<details>
<summary>Hint</summary>

```python
def category(product):
    p = clean_text(product)
    ...
```

Use a dict + `.get(cat, 0) + 1` to count.
</details>

### Exercise 3 — One-line clean (comprehensions)
In **one** list comprehension (plus `clean_row` if you want), build `east_big` = list of cleaned rows where `Region == "East"` and `Total >= 100`. Then in one dict comprehension build `{OrderID: Total}` from it.
*Expected: Orders 102 (127.5), 108 (150.0), and 110 (199.96) — note 110’s Region is `"east"`, which `clean_text` normalises to `"East"`. Hint: `[clean_row(r) for r in sales if ...]` — but filter on cleaned values, so either clean twice or clean first then filter.*

**Follow-up:** Confirm east_big sums to East's regional total. Check: 477.46.

<details>
<summary>Hint</summary>

Easiest readable answer is two steps: `cleaned = [clean_row(r) for r in sales]` then `east_big = [r for r in cleaned if ...]`. One-liner is possible but two lines is more Pythonic — say so in a comment.
</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
big = []
for r in sales:
    qty, price = r["Quantity"], r["Price"]
    if qty is None or price is None:
        continue
    total = qty * price
    if total > 500:
        big.append((r["OrderID"], round(total, 2)))
        print(r["OrderID"], round(total, 2))
print("count:", len(big))  # 101 1799.98 / 106 899.99 / count: 2

# --- Solution 2 ---
def category(product):
    p = clean_text(product)
    if p == "":
        return "Unknown"
    if p in ("Laptop", "Monitor", "Keyboard", "Mouse"):
        return "Computer"
    if p == "Headset":
        return "Audio"
    return "Other"

counts = {}
for r in sales:
    cat = category(r["Product"])
    counts[cat] = counts.get(cat, 0) + 1
print(counts)  # {'Computer': 10, 'Unknown': 1, 'Audio': 1}

# --- Solution 3 ---
cleaned = [clean_row(r) for r in sales]
east_big = [r for r in cleaned if r["Region"] == "East" and r["Total"] >= 100]
east_map = {r["OrderID"]: r["Total"] for r in east_big}
print(east_big)
print(east_map)  # {102: 127.5, 108: 150.0, 110: 199.96}

# --- Follow-up 1 ---
valid = [r for r in sales if str(r["Product"]).strip()
         and r["Quantity"] is not None and r["Price"] is not None]
valid_rev = sum(r["Quantity"] * r["Price"] for r in valid)
share = round(sum(t for _, t in big) / valid_rev, 4)
print(share)  # ~0.7184 — two orders drive ~72% of valid revenue
assert abs(share - 0.7184) < 0.001

# --- Follow-up 2 ---
cshare = round(counts["Computer"] / len(sales), 4)
print(cshare)  # 0.8333
assert cshare == 0.8333

# --- Follow-up 3 ---
etot = round(sum(east_map.values()), 2)
print(etot)  # 477.46 == East regional total
assert etot == 477.46


### What to learn next
- `csv.DictReader` / `csv.DictWriter` to load/save the real file without pandas.
- `pathlib`, `datetime` for file dates.
- Then pandas: `DataFrame` = supercharged list-of-dicts.
- Cheat sheet: `list` → column, `dict` → row, `for` → row-by-row, `def` → reusable step, `[... for ... if ...]` → one-line transform.

*Files in this folder: `sales_spreadsheet.csv` (messy input) + this notebook in Markdown. Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
